# Intuition test: unsupervised learning

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Covers notebooks 01 to 06. Seeded and self-contained.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common.datasets import (
    load_digits_data,
    make_blobs_data,
    make_customer_data,
    make_moons_data,
)

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

## Q1. k-means on moons

Two interleaved half-moons, 300 points, a little noise. You run k-means with $k=2$. The adjusted Rand index (ARI) is 1 for a perfect match with the two moons and about 0 for a random split. What do you expect, and what does the boundary between the two clusters look like?

Prediction: *(write here)*

In [ ]:
Xm_df, ym = make_moons_data(300, noise=0.08, seed=SEED)
Xm = Xm_df.to_numpy()
km = KMeans(2, n_init=10, random_state=SEED).fit(Xm)
print("ARI:", round(adjusted_rand_score(ym, km.labels_), 3))
print("cluster centres:\n", km.cluster_centers_)

## Q2. Inertia as k grows

You fit k-means on 300 points for $k=1,2,\dots,300$. What happens to the inertia as $k$ grows, and what is its value at $k=300$? Can you pick $k$ by taking the one with the lowest inertia?

Prediction: *(write here)*

In [ ]:
Xb_df, _ = make_blobs_data(300, centers=4, cluster_std=0.9, seed=SEED)
Xb = Xb_df.to_numpy()
for k in (1, 2, 4, 8, 50, 300):
    print(f"k={k:>3}: inertia {KMeans(k, n_init=3, random_state=SEED).fit(Xb).inertia_:.2f}")

## Q3. Scaling before clustering

The customer data have annual spend (in the thousands) and discount share (between 0 and 1) among five features, and four true segments. You run k-means with $k=4$ on the raw table, then on the standardised table. How do the ARI scores compare against the true segments?

Prediction: *(write here)*

In [ ]:
Xc, seg = make_customer_data(n_samples=1000, seed=SEED)
raw = KMeans(4, n_init=10, random_state=SEED).fit(Xc)
scaled = KMeans(4, n_init=10, random_state=SEED).fit(StandardScaler().fit_transform(Xc))
print("ARI raw:   ", round(adjusted_rand_score(seg, raw.labels_), 3))
print("ARI scaled:", round(adjusted_rand_score(seg, scaled.labels_), 3))

## Q4. DBSCAN and eps

On the same moons, DBSCAN with `min_samples=5`. You try `eps` of 0.05, 0.2 and 1.0. For each, roughly how many clusters and how many noise points do you expect?

Prediction: *(write here)*

In [ ]:
for eps in (0.05, 0.2, 1.0):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    print(
        f"eps={eps}: {len(set(lab) - {-1})} clusters, {int((lab == -1).sum())} noise points, ARI {adjusted_rand_score(ym, lab):.2f}"
    )

## Q5. When does a Gaussian mixture beat k-means?

Two elongated, tilted clusters side by side, 400 points each, closer together than they are long. Which method do you expect to recover them better, k-means or a two-component Gaussian mixture, and why?

Prediction: *(write here)*

In [ ]:
cov = np.array([[4.0, 3.6], [3.6, 4.0]])  # a long thin diagonal ellipse
A = rng.multivariate_normal([0, 0], cov, 400)
B = rng.multivariate_normal([0, 4], cov, 400)
Xe, ye = np.vstack([A, B]), np.r_[np.zeros(400), np.ones(400)]
print(
    "k-means ARI:",
    round(adjusted_rand_score(ye, KMeans(2, n_init=10, random_state=SEED).fit_predict(Xe)), 3),
)
gm = GaussianMixture(2, covariance_type="full", n_init=5, random_state=SEED).fit(Xe)
print("GMM ARI:    ", round(adjusted_rand_score(ye, gm.predict(Xe)), 3))

## Q6. How many PCA components for 95%?

The 8 by 8 digits have 64 pixel columns. How many principal components do you need to keep 95% of the variance: about 5, 15, 30 or 50? And what share of the variance does PC1 alone carry?

Prediction: *(write here)*

In [ ]:
Xd, yd = load_digits_data()
p = PCA(random_state=SEED).fit(Xd)
cum = np.cumsum(p.explained_variance_ratio_)
print("components for 95%:", int(np.argmax(cum >= 0.95)) + 1)
print("PC1 share:", p.explained_variance_ratio_[0].round(3), " PC1 + PC2:", cum[1].round(3))

## Q7. Variance is not relevance

One feature is pure noise with a large spread (standard deviation 10). The other separates two classes cleanly but has a small spread. You reduce to one PCA component and then classify with kNN. What accuracy do you expect? And with two components?

Prediction: *(write here)*

In [ ]:
n = 600
labels = rng.integers(0, 2, n)
Xs = np.c_[rng.normal(0, 10, n), labels * 1.5 + rng.normal(0, 0.3, n)]
for k in (1, 2):
    m = make_pipeline(PCA(k, random_state=SEED), KNeighborsClassifier(5))
    print(f"{k} component(s): CV accuracy {cross_val_score(m, Xs, labels, cv=5).mean():.3f}")

## Q8. t-SNE and cluster size

Three clusters of 150 points in 10 dimensions, with true spreads of 0.3, 1 and 3 (a tenfold range). In a t-SNE map, will the widest cluster look about ten times as wide as the tightest? What does this say about reading cluster size from a t-SNE plot?

Prediction: *(write here)*

In [ ]:
centres = np.eye(10)[:3] * 12
spreads = (0.3, 1.0, 3.0)
Xg = np.vstack([rng.normal(c, s, (150, 10)) for c, s in zip(centres, spreads)])
yg = np.repeat([0, 1, 2], 150)
emb = TSNE(2, perplexity=30, init="pca", random_state=SEED).fit_transform(Xg)
for j, s in enumerate(spreads):
    print(f"true spread {s}: map std {emb[yg == j].std(axis=0).mean():.2f}")

## Answers

Check each against your prediction.

**Q1.** ARI is about 0.26, far from 1. The two centres sit at about (-0.2, 0.6) and (1.2, -0.1), and the boundary is a straight line halfway between them. That line cuts both moons. k-means assumes round clusters, and a moon is a curve, so the closest-centre rule cannot follow it.

**Q2.** Inertia falls every time $k$ rises and is exactly 0 at $k=300$, where every point is its own cluster. So the lowest inertia always picks the largest $k$ and tells you nothing. Look for the elbow, where the fall flattens (here at $k=4$, the number of blobs, with inertia 459 against 9,308 at $k=2$), or use silhouette or a stability check.

**Q3.** Raw: about 0.57. Scaled: about 0.99. Without scaling, annual spend (thousands) accounts for essentially all of the squared distance, and the other four features hardly matter. After standardising each feature counts equally.

**Q4.** At 0.05 most points are noise (about 280 of 300) with a few small clusters. At 0.2 there are 2 clusters and no noise, which matches the moons. At 1.0 everything joins into one cluster. Only a narrow band of `eps` works, which is why the k-distance plot is worth drawing.

**Q5.** The mixture, by a wide margin (ARI 0.96 against 0.25). k-means splits by the closest centre, and with elongated clusters that stack side by side the line halfway between centres cuts across the long direction of each ellipse. The mixture learns a tilted covariance for each component, so its boundary follows the ellipses. When clusters are round and well separated the two agree and k-means is faster.

**Q6.** 29 components for 95% (less than half of 64). PC1 alone carries about 15% and PC1 plus PC2 about 29%. Pixels in a digit are strongly correlated, so a few directions describe most of the variation, but the variance is spread more evenly than in iris, where PC1 holds 92%.

**Q7.** One component: about 48%, which is chance. PCA keeps the noise column because it has the larger variance and throws away the one that carries the signal. Two components: about 97%. PCA never sees the labels. When the task is supervised, choose $k$ by validation score, not by variance.

**Q8.** No. The map spreads are nearly the same for all three (about 3.1 each), although the true spreads differ tenfold. t-SNE picks each point's bandwidth to hit the same perplexity, so dense regions are expanded and loose ones shrunk. Cluster size on a t-SNE plot is not evidence of cluster size in the data, and the distances between clusters are not reliable either. Use the plot for which points are near which, then check anything important in the original space.